# Tâche 9: Rapport qualité et statistiques par langue

**Projet 1 · African Language Data & LLM Readiness · AfrikLang**

Agrège les bilans déjà produits par les tâches 6 (normalisation), 7 (contrôle qualité) et 8 (export) en :
- `data/processed/qualite/rapport_qualite_par_langue.csv` / `.jsonl` un tableau, une ligne par langue
- `data/processed/qualite/couverture.json` périmètre cible vs langues réellement dans le corpus final


In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Users\akota\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def lancer(*args):
    # Lance src/rapport_qualite.py et affiche son journal au fur et à mesure.
    env = dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1")
    p = subprocess.Popen([sys.executable, "-u", str(ROOT / "src" / "rapport_qualite.py"), *args], cwd=ROOT,
                         env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding="utf-8", errors="replace")
    try:
        for ligne in p.stdout:
            print(ligne, end="")
    except KeyboardInterrupt:
        p.terminate()
        print("Arrêté.")
    return p.wait()

## Lancer le rapport

In [12]:
lancer()
QUALITE = ROOT / "data" / "processed" / "qualite"
tab = pd.read_csv(QUALITE / "rapport_qualite_par_langue.csv")
tab.set_index("langue")

69 langue(s) ; écrit dans data\processed\qualite
couverture : 65/68 (95.6 %)
  à 0 texte final : aa, hz, kr
manifeste de collecte : 69 langue(s) (source : C:\Users\akota\Desktop\projet_stage_nlp\data\processed\collecte\20261105T065025Z)


,textes_collectes,textes_normalises,textes_modifies_normalisation,near_duplicate,template_generated,boilerplate,language_mismatch,au_moins_un_drapeau,plusieurs_drapeaux,erreurs_collecte_detectees,textes_ecartes_export,textes_exportes_final,textes_corpus_final,longueur_mediane_caracteres,textes_courts_moins_200_caracteres,textes_courts_moins_500_caracteres
langue,,,,,,,,,,,,,,,,
aa,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0
af,131092,131092,36008,8,8930,75,17240,19646,6607,1,1,131091,131091,737.0,17928,49967
ak,1,1,0,0,0,0,1,1,0,0,0,1,1,105.0,1,1
am,16286,16286,2538,14,2353,13,2037,2535,1882,0,0,16286,16286,108.0,10626,13055
ann,494,494,25,0,0,0,25,25,0,0,0,494,494,614.5,83,212
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
wo,1809,1809,270,10,0,0,118,128,0,0,0,1809,1809,284.0,788,1113
xh,3090,3090,579,2,0,0,126,128,0,0,0,3090,3090,735.0,669,1187
yo,41803,41803,3403,21,368,0,1517,1869,37,0,0,41803,41803,66.0,26392,30240


## Couverture du périmètre

In [10]:
couverture = json.loads((QUALITE / "couverture.json").read_text(encoding="utf-8"))
print(f"{couverture['langues_couvertes']}/{couverture['perimetre_cible']} "
      f"({couverture['couverture_pourcentage']} %)")
print("à 0 texte final :", couverture["langues_a_zero_texte_final"] or "aucune")
print("jamais traitées :", couverture["langues_cibles_jamais_traitees"] or "aucune")

65/68 (95.6 %)
à 0 texte final : ['aa', 'hz', 'kr']
jamais traitées : aucune


## Manifeste de collecte (tâche 9.2)

Produit automatiquement par la même cellule `lancer()` ci-dessus si `data/processed/collecte/<run>/` est accessible (sinon cette section est vide — relancer après avoir déposé le dossier de collecte, ou avec `lancer("--collecte", "CHEMIN")`).


In [11]:
chemin_manifeste = QUALITE / "manifeste_collecte.csv"
if chemin_manifeste.exists():
    manifeste = pd.read_csv(chemin_manifeste)
    display(manifeste.set_index("langue"))
else:
    print("Pas encore généré — voir le message de lancer() ci-dessus.")

Pas encore généré — voir le message de lancer() ci-dessus.


## Vues utiles pour le rapport

In [5]:
# Langues avec le plus de doublons/anomalies, en proportion du volume collecté
tab["part_au_moins_un_drapeau"] = (tab["au_moins_un_drapeau"] / tab["textes_collectes"]).round(3)
tab.sort_values("part_au_moins_un_drapeau", ascending=False)[
    ["langue", "textes_collectes", "au_moins_un_drapeau", "part_au_moins_un_drapeau",
     "textes_exportes_final"]].head(15)

,langue,textes_collectes,au_moins_un_drapeau,part_au_moins_un_drapeau,textes_exportes_final
2,ak,1,1,1.000,1
35,kus,1721,1620,0.941,1721
61,tum,19880,13793,0.694,19880
30,kg,2256,1510,0.669,2256
40,ng,20,12,0.600,20
51,shi,10952,6350,0.580,10951
6,ary,20321,10952,0.539,20321
67,zgh,12192,6361,0.522,12192
38,mg,104097,53498,0.514,104097
5,ar,1332978,432469,0.324,1332977


In [6]:
# Volume final (texte exporté) par langue, trié décroissant
tab.sort_values("textes_exportes_final", ascending=False)[
    ["langue", "textes_exportes_final", "longueur_mediane_caracteres"]].head(15)

,langue,textes_exportes_final,longueur_mediane_caracteres
7,arz,1631258,221.0
5,ar,1332977,284.0
1,af,131091,737.0
56,sw,129081,341.0
21,ha,123650,1899.0
38,mg,104097,278.0
17,fr,99961,1319.0
23,ig,56485,1611.0
66,yo,41803,66.0
15,ff,34106,948.0
